LLM - huggingface LLM (default : gpt-3.5-turbo)
https://docs.llamaindex.ai/en/stable/module_guides/models/llms/usage_custom/

In [1]:
pip install llama-index-llms-ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama

Settings.llm = Ollama(model="llama3", request_timeout=360.0)

EMBEDDING MODEL - BAAI (default : text-embedding-ada-002)
https://docs.llamaindex.ai/en/stable/module_guides/models/embeddings/

In [3]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")

VECTOR STORE - 수정 X (customize하려면 pinecone 써야함)

In [4]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)

INDEX SETUP

In [5]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [6]:
from llama_index.core.indices import SummaryIndex

summary_index = SummaryIndex.from_documents(documents)

QUERY ENGINE

In [7]:
query_engine = index.as_query_engine()

EVALUATION

In [8]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [9]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [10]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [11]:
responses_str = []
responses = []
count=0

In [12]:
for question in questions:
    count+=1
    query = f" Determine whether the statement is 'True' or 'False'. {question}. Please answer with 'True' or 'False' at the beginning of your response."
    response = query_engine.query(query)
    responses.append(response)
    
    response_str=str(response)
    print(count, response) 
    if "True" in response_str:
        response_str="True"
    elif "False" in response_str:
        response_str="False"
    else:
        print("error")
    responses_str.append(response_str)

1 True

According to the context information, the concept of a fourth dimension extends beyond our three-dimensional perception, incorporating length, width, and height into a framework that includes an additional spatial dimension. This notion finds its roots in advanced theoretical physics, particularly in string theory and higher-dimensional space concepts.
2 False

The context information suggests that while the discovery of a fourth dimension could lead to the creation of a breathable liquid atmosphere on Earth, humans would still need adaptations or devices to survive underwater. The text mentions that terrestrial organisms, including humans, would need to evolve mechanisms to extract oxygen from the liquid atmosphere, potentially akin to gills in aquatic species. This implies that humans would require some form of respiratory adaptation or device to breathe underwater without suffering from hypoxia.
3 True
4 False

The context information does not explicitly state that the fourt

In [13]:
correct_count=0
number=0

for response, answer, response_str, question in zip(responses, answers, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 9 Fourth-dimensional technology will not affect the efficiency of oxygen delivery in the liquid atmosphere. (True/False)
RESPONSE True
CORRECT ANSWER False

<<wrong>>
 31 The extreme gravitational time dilation near a neutron star would have no effect on long-term projects. (True/False)
RESPONSE True
Living near a neutron star would expose the inhabitants to extreme gravitational time dilation, where time would pass more slowly relative to Earth. This effect could create intriguing possibilities for long-term projects and interstellar travel, as significant periods could elapse on Earth while only a few years pass on the space station.
CORRECT ANSWER False

<<wrong>>
 32 Fourth-dimensional technology will have significant geopolitical implications due to its potential. (True/False)
RESPONSE False.

The context does not explicitly mention fourth-dimensional technology having significant geopolitical implications due to its potential. While it discusses the profound impact on 

In [16]:
accuracy = (correct_count / len(questions)) * 100

In [17]:
print(f"model_name: ", Settings.llm.model)
print(f"Embedding Model Name: {Settings.embed_model.model_name}")
print()

print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

model_name:  llama3
Embedding Model Name: BAAI/bge-base-en-v1.5

Total Questions: 75
Correct Answers: 55
Accuracy: 73.33%
